# Multimodal Feature Fusion Practice

**Topic:** Multimodal Learning &nbsp;·&nbsp; **Difficulty:** Hard

## The data (created in the next cell)

- `img_train` `(4000, 12, 12)`: grayscale product photos containing one of 3 shapes (bar / disc / cross) with noise and jitter.
- `tab_train` `(4000, 6)`: spec-sheet features. A hidden linear rule splits them into pass/fail.
- `y_train` `(4000,)` holds six classes: `label = shape_index * 2 + spec_pass`.
- `img_test`, `tab_test` (1,200 each): predict these. The evaluation cell holds the labels.

## Your task

1. Try each modality alone first: measure the ceiling you hit.
2. Fuse: concatenate flattened image pixels (or CNN features) with the tabular columns and train one classifier, or build a two-branch PyTorch network that merges embeddings.
3. Set `predictions` to your 0–5 labels for the test set.

## Evaluation

Test accuracy must reach the target shown in the evaluation cell.


In [ ]:
import numpy as np

rng = np.random.default_rng(541)
w_spec = rng.normal(0, 1.0, 6)

def make(n, seed):
    r = np.random.default_rng(seed)
    imgs = r.normal(0.15, 0.08, (n, 12, 12)).clip(0, 1).astype(np.float32)
    shapes = r.integers(0, 3, n)
    for i in range(n):
        cy, cx = r.integers(4, 8, 2)
        b = 1.0 - r.uniform(0, 0.25)
        if shapes[i] == 0:                      # horizontal bar
            imgs[i, cy, cx-3:cx+4] = b
            imgs[i, cy+1, cx-3:cx+4] = b
        elif shapes[i] == 1:                    # disc
            yy, xx = np.mgrid[0:12, 0:12]
            imgs[i][(yy - cy) ** 2 + (xx - cx) ** 2 <= 5] = b
        else:                                   # cross
            imgs[i, cy-3:cy+4, cx] = b
            imgs[i, cy, cx-3:cx+4] = b
    tab = r.normal(size=(n, 6))
    spec_pass = (tab @ w_spec + r.normal(0, 0.25, n) > 0).astype(int)
    y = shapes * 2 + spec_pass
    return imgs, tab, y

img_train, tab_train, y_train = make(4000, 551)
img_test, tab_test, y_test = make(1200, 552)
print(f"images {img_train.shape}, tabular {tab_train.shape}, 6 classes")
_GRADE = {"y_test": y_test}
del y_test


In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.





In [ ]:
# =====================================================================
# EVALUATION: run this cell to compare your predictions to the truth
# =====================================================================
import numpy as np

TARGET_ACCURACY = 0.75

for _name in ("predictions", "_GRADE"):
    if _name not in dir():
        raise NameError(f"`{_name}` is not defined. Run the cells above first, then your solution cell.")

_pred = np.asarray(predictions).ravel()
_true = np.asarray(_GRADE["y_test"]).ravel()
if _pred.shape != _true.shape:
    raise ValueError(f"`predictions` has shape {_pred.shape}, expected {_true.shape}.")

accuracy = float((_pred == _true).mean())
print("=" * 52)
print(f"Test accuracy : {accuracy:.4f}")
print(f"Target        : >= {TARGET_ACCURACY:.2f}")
print("-" * 52)
print("Per-class recall:")
for _cls in np.unique(_true):
    _mask = _true == _cls
    print(f"  class {int(_cls):>6} : {float((_pred[_mask] == _cls).mean()):.3f}   ({int(_mask.sum())} test samples)")
print("=" * 52)
if accuracy >= TARGET_ACCURACY:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet: revisit your model and try again.")


## Solution

Neither modality alone can produce all six labels: the shape lives in the pixels and the pass/fail bit lives in the spec sheet. Concatenate flattened pixels with the six tabular columns and train one classifier on the fused features.

```python
import numpy as np
from sklearn.neural_network import MLPClassifier

fused_train = np.hstack([img_train.reshape(len(img_train), -1), tab_train])
fused_test = np.hstack([img_test.reshape(len(img_test), -1), tab_test])

clf = MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=500, random_state=0)
clf.fit(fused_train, y_train)
predictions = clf.predict(fused_test)
```
